In [0]:
from pyspark.sql import functions as F

from delta.tables import DeltaTable
from pyspark.sql import functions as F

catalogo = "databricks_cata_managed"

tabela_silver = f"{catalogo}.silver.cotacao_moeda"
tabela_gold_fato = f"{catalogo}.gold.fato_cotacao_diaria"
tabela_gold_indicador = f"{catalogo}.gold.fato_indicador_moeda"

df_silver = spark.table(tabela_silver)
df_silver = spark.table(f"{catalogo}.silver.cotacao_moeda")

In [0]:
from pyspark.sql import functions as F

from delta.tables import DeltaTable
from pyspark.sql import functions as F

catalogo = "databricks_cata_managed"

tabela_silver = f"{catalogo}.silver.cotacao_moeda"
tabela_gold_fato = f"{catalogo}.gold.fato_cotacao_diaria"
tabela_gold_indicador = f"{catalogo}.gold.fato_indicador_moeda"

df_silver = spark.table(tabela_silver)
df_silver = spark.table(f"{catalogo}.silver.cotacao_moeda")

df_gold = (
    df_silver
    .groupBy(
        "codigo_moeda",
        "data_cotacao"
    )
    .agg(
        F.count("*").alias("qtd_cotacoes"),

        F.avg("cotacao_compra").cast("decimal(18,6)").alias("media_compra"),
        F.avg("cotacao_venda").cast("decimal(18,6)").alias("media_venda"),

        F.min("cotacao_compra").alias("menor_compra"),
        F.max("cotacao_compra").alias("maior_compra"),

        F.min("cotacao_venda").alias("menor_venda"),
        F.max("cotacao_venda").alias("maior_venda")
    )
    .withColumn("_data_atualizacao", F.current_timestamp())
)

if not spark.catalog.tableExists(tabela_gold_fato):
    (
        df_gold.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(tabela_gold_fato)
    )
else:
    delta_gold = DeltaTable.forName(spark, tabela_gold_fato)

    (
        delta_gold.alias("destino")
        .merge(
            df_gold.alias("origem"),
            """
            destino.codigo_moeda = origem.codigo_moeda
            AND destino.data_cotacao = origem.data_cotacao
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )